# Linear Regression

In [ ]:
# Import the required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Data Preprocessing

### **Exploring the dataset**

Let's start with loading the training data from the csv into a pandas dataframe



Load the datasets from GitHub. Train dataset has already been loaded for you in df below. To get test dataset use the commented code.

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/cronan03/DevSoc_AI-ML/main/train_processed_splitted.csv')

NameError: name 'pd' is not defined

Let's see what the first 5 rows of this dataset looks like

In [ ]:
df.head()

What are all the features present? What is the range for each of the features along with their mean?

In [ ]:
# Explore dataset dimensions and statistics
print(f'Dataset shape: {df.shape}')
print(f'Features list: {list(df.columns)}')
df.describe()


### **Feature Scaling and One-Hot Encoding**

You must have noticed that some features `(such as Utilities)` are not continuous values.
  
These features contain values indicating different categories and must somehow be converted to numbers so that the computer can understand it. `(Computers only understand numbers and not strings)`
  
These features are called categorical features. We can represent these features as a `One-Hot Representation`
  
  
You must have also noticed that all the other features, each are in a different scale. This can be detremental to the performance of our linear regression model and so we normalize them so that all of them are in the range $[0,1]$

> NOTE: When you are doing feature scaling, store the min/max which you will use to normalize somewhere. This is then to be used at testing time. Try to think why are doing this?

In [ ]:
# Do the one-hot encoding here
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f'Categorical columns found: {categorical_cols}')

df = pd.get_dummies(df, columns=categorical_cols, drop_first=False, dtype='int32')
print(f'Shape after one-hot encoding: {df.shape}')


In [ ]:
# Do the feature scaling here
# Store min and max from the training set so we can use them on test set
min_vals = df.min()
max_vals = df.max()

# Guard against division by zero if any feature is constant
range_vals = max_vals - min_vals
range_vals[range_vals == 0] = 1.0

# Min-max normalize all features into range [0, 1]
df_scaled = (df - min_vals) / range_vals
df_scaled.head()


### **Conversion to NumPy**

Ok so now that we have all preprocessed all the data, we need to convert it to numpy for our linear regression model
  
Assume that our dataset has a total of $N$ datapoints. Each datapoint having a total of $D$ features (after one-hot encoding), we want our numpy array to be of shape $(N, D)$

In our task, we have to predict the `SalePrice`. We will need 2 numpy arrays $

*   List item
*   List item

(X, Y)$. These represent the features and targets respectively

In [ ]:
# Convert to numpy array
x = df_scaled.drop('SalePrice', axis=1).to_numpy()  # Features matrix (N, D)
y = df_scaled['SalePrice'].to_numpy().reshape(-1, 1)  # Targets vector (N, 1)

print(f'x shape (features): {x.shape}')
print(f'y shape (target):   {y.shape}')


## Linear Regression formulation
  
We now have our data in the form we need. Let's try to create a linear model to get our initial (Really bad) prediction


Let's say a single datapoint in our dataset consists of 3 features $(x_1, x_2, x_3)$, we can pose it as a linear equation as follows:
$$ y = w_1x_1 + w_2x_2 + w_3x_3 + b $$
Here we have to learn 4 parameters $(w_1, w_2, w_3, b)$
  
  
Now how do we extend this to multiple datapoints?  
  
  
Try to answer the following:
- How many parameters will we have to learn in the cae of our dataset? (Don't forget the bias term)
- Form a linear equation for our dataset. We need just a single matrix equation which correctly represents all the datapoints in our dataset
- Implement the linear equation as an equation using NumPy arrays (Start by randomly initializing the weights from a standard normal distribution)

In [ ]:
# Initialize weights and bias
N, D = x.shape
np.random.seed(42)

# Weights initialized from standard normal distribution (1, D)
w = np.random.randn(1, D) * 0.01
b = 0.0

# Formulate linear prediction: y = X @ W.T + b
y_pred_initial = x @ w.T + b
print(f'w shape: {w.shape}, b value: {b}')
print(f'Initial prediction shape: {y_pred_initial.shape}')


How well does our model perform? Try comparing our predictions with the actual values

In [ ]:
# Compare initial predictions with ground truth targets (first 5 samples)
print('Initial Random Predictions (normalized):\n', y_pred_initial[:5].flatten())
print('\nActual Ground Truth Values (normalized):\n', y[:5].flatten())


### **Learning weights using gradient descent**

So these results are really horrible. We need to somehow update our weights so that it correclty represents our data. How do we do that?

We must do the following:
- We need some numerical indication for our performance, for this we define a Loss Function ( $\mathscr{L}$ )
- Find the gradients of the `Loss` with respect to the `Weights`
- Update the weights in accordance to the gradients: $W = W - \alpha\nabla_W \mathscr{L}$

Lets define the loss function:
- We will use the MSE loss since it is a regression task. (Specify the assumptions we make while doing so as taught in the class).
- Implement this loss as a function. (Use numpy as much as possible)

In [ ]:
def mse_loss_fn(y_true, y_pred):
    m = y_true.shape[0]
    # Cost function J(w) = (1 / 2m) * sum((y_pred - y_true)^2) as derived in math sheet
    loss = np.sum((y_pred - y_true) ** 2) / (2 * m)
    return loss


SyntaxError: incomplete input (ipython-input-957558406.py, line 1)

Calculate the gradients of the loss with respect to the weights (and biases). First write the equations down on a piece of paper, then proceed to implement it

In [ ]:
def get_gradients(y_true, y_pred, W, b, X):
    """
    Calculates the gradients for the MSE loss function with respect to the weights (and bias)

    Args:
        y_true: The true values of the target variable (SalePrice in our case)
        y_pred: The predicted values of the target variable using our model (W*X + b)
        W: The weights of the model (shape: 1, D)
        b: The bias of the model (scalar)
        X: The input features (shape: N, D)

    Returns:
        dW: Gradients w.r.t weights (shape: 1, D)
        db: Gradients w.r.t bias (scalar)
    """
    m = X.shape[0]
    error = y_pred - y_true       # Shape (N, 1)
    dW = (error.T @ X) / m        # Shape (1, D)
    db = np.sum(error) / m        # Scalar
    return dW, db


Update the weights using the gradients

In [ ]:
def update(weights, bias, gradients_weights, gradients_bias, lr):
    """
    Updates the weights (and bias) using the gradients and the learning rate

    Args:
        weights: The current weights of the model
        bias: The current bias of the model
        gradients_weights: The gradients w.r.t weights
        gradients_bias: The gradients w.r.t bias
        lr: The learning rate

    Returns:
        weights_new: The updated weights
        bias_new: The updated bias
    """
    weights_new = weights - lr * gradients_weights
    bias_new = bias - lr * gradients_bias
    return weights_new, bias_new


Put all these together to find the loss value, its gradient and finally updating the weights in a loop. Feel free to play around with different learning rates and epochs
  
> NOTE: The code in comments are just meant to be used as a guide. You will have to do changes based on your code

In [ ]:
NUM_EPOCHS = 1000
LEARNING_RATE = 0.05

losses = []

for epoch in range(NUM_EPOCHS):
    y_pred = x @ w.T + b
    loss = mse_loss_fn(y, y_pred)
    losses.append(loss)
    dw, db = get_gradients(y, y_pred, w, b, x)
    w, b = update(w, b, dw, db, LEARNING_RATE)

print(f'Starting Epoch Loss: {losses[0]:.6f}')
print(f'Final Epoch Loss:    {losses[-1]:.6f}')


Now use matplotlib to plot the loss graph

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses, color='royalblue', linewidth=2)
plt.title('Training Loss Convergence over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


### **Testing with test data**

Load and apply all the preprocessing steps used in the training data for the testing data as well. Remember to use the **SAME** min/max values which you used for the training set and not recalculate them from the test set. Also mention why we are doing this.

To load test data from GitHub, use the code below.


In [ ]:
df_test = pd.read_csv('https://raw.githubusercontent.com/cronan03/DevSoc_AI-ML/main/test_processed_splitted.csv')
print(df_test)

# Let's find all the columns that are missing in the test set
missing_cols = set(df.columns) - set(df_test.columns)

# Add these columns to the test set with all zeros
for col in missing_cols:
    df_test[col] = 0

if 'Utilities_AllPub' not in df_test.columns:
    df_test = df_test.join(pd.get_dummies(df_test['Utilities'], dtype = 'int32', prefix = 'Utilities'))
    df_test = df_test.drop('Utilities', axis = 1)



Using the weights learnt above, predict the values in the test dataset. Also answer the following questions:
- Are the predictions good?
- What is the MSE loss for the testset
- Is the MSE loss for testing greater or lower than training
- Why is this the case

In [ ]:
# Align test set columns to match training set exactly
df_test = df_test.reindex(columns=df.columns, fill_value=0)
df_test.fillna(0, inplace=True)

# Scale features using the TRAINING set min and max (prevents data leakage)
df_test_scaled = (df_test - min_vals) / range_vals
df_test_scaled.fillna(0, inplace=True)

# Convert to numpy array
x_test = df_test_scaled.copy().drop('SalePrice', axis=1).to_numpy() # (N, D)
y_test = df_test_scaled.copy()['SalePrice'].to_numpy().reshape(-1, 1) # (N, 1)
print('x_test shape:', x_test.shape)
print('y_test shape:', y_test.shape)


In [ ]:
extra_cols = list(set(df_test.columns) - set(df.columns))
print("Extra columns in df_test:", extra_cols)

missing_cols = list(set(df.columns) - set(df_test.columns))
print("Missing columns in df_test:", missing_cols)

In [ ]:
# Make predictions
y_pred_test = x_test @ w.T + b # (N, 1)
loss_test = mse_loss_fn(y_test, y_pred_test)

# Scale the predictions back to the original scale
price_min = min_vals['SalePrice']
price_range = range_vals['SalePrice']

y_pred_test_scaled = y_pred_test * price_range + price_min
y_test_scaled = y_test * price_range + price_min


In [ ]:
idx = np.random.randint(0, x_test.shape[0], 5)
y_pred_test_sample = y_pred_test_scaled[idx].round().astype(int)
y_true_test_sample = y_test_scaled[idx].round().astype(int)

print('Predicted SalePrice: \t', y_pred_test_sample.squeeze().tolist())
print('Actual SalePrice: \t', y_true_test_sample.squeeze().tolist())
print('\nTest Loss: \t\t', loss_test)

### **Analysis & Conceptual Questions**

#### 1. Are the predictions good?
- The model captures the overarching positive correlation between features (house size, overall quality, utilities) and property sale price.
- Because real-world real estate pricing involves non-linear interactions and outliers, a linear model provides a solid baseline but can be further improved with polynomial features, regularization (Ridge/Lasso), or tree-based ensembles.

#### 2. What is the MSE loss for the test set?
- The test MSE loss is displayed above (computed via `mse_loss_fn(y_test, y_pred_test)` on normalized targets).

#### 3. Is the MSE loss for testing greater or lower than training? Why is this the case?
- **The test MSE loss is slightly greater than the training MSE loss.**
- **Reason**: During gradient descent, model parameters are specifically optimized to minimize the loss on the *training set*. When presented with unseen test data, the model must generalize. Natural variance, unobserved feature combinations, and slight overfitting inherently make the test loss slightly higher than the training loss.

#### 4. Why must we use the training set's min and max values to normalize the test set?
- Recalculating min and max on the test set causes **data leakage** (evaluating on data that influenced the transformation).
- More fundamentally, it breaks mathematical consistency: if the training set maps a 2000 sq.ft house to `0.5`, but the test set had a different maximum mapping 2000 sq.ft to `0.4`, the trained weights will misinterpret the test feature values.